BIDS datastructure restructuring

In [1]:
import os
import re
from pathlib import Path
import shutil

In [2]:
# Alter BIDS
OLD_ROOT = Path("/mnt_01/ds-ASD/BIDS_old/ds-asd")
# Neuer BIDS
NEW_ROOT = Path("/mnt_01/ds-ASD")

In [3]:
# Formatierung der Subjekt-IDs
def short_sub(sub_str: str) -> str:
    """Formatiert die Subjekt-ID in die Kurzform, z.B. 'sub-00001' -> sub-'01'."""
    match = re.search(r"sub-(\d+)", sub_str)
    number = int(match.group(1)) if match else None 
    return f"sub-{number:02d}" if number is not None else sub_str
    

In [ ]:
#Formatierung der Session-IDs
def short_ses(ses_str: str) -> str:
    """Formatiert die Session-ID in die Kurzform, z.B. 'ses-00001' -> 'ses-1'."""
    match = re.search(r"ses-(\d+)", ses_str)
    number = int(match.group(1)) if match else None
    return f"ses-{number}" if number is not None else ses_str


Func

In [ ]:
def rename_func(name: str, sub_new: str, ses_new: str) -> str:
    """Ersetzt die Subjekt- und Session-IDs, sowie Task im Dateinamen und entfernt acq/rec."""
    
    # sub-00001 -> sub-01
    name = re.sub(r"sub-\d+", sub_new, name)
    
    # ses-00001 -> ses-1
    name = re.sub(r"ses-\d+", ses_new, name)
    
    # task-fmri -> task-chase
    name = name.replace("task-fmri", "task-chase")

    # acq/rec entfernen
    name = re.sub(r"_acq-[^_\.]+", "", name)
    name = re.sub(r"_rec(?:-[^_\.]+)?", "", name)

    # Falls im Quellnamen nach task- direkt eine Zahl steht, setze explizit "run-" ein
    name = re.sub(r"(task-[^_]+)-(\d+)(?=_)", r"\1_run-\2", name)
                  
    # Underscores glätten
    # z.B. sub-01_ses-1_task-chase__run-1_bold.nii.gz -> sub-01_ses-1_task-chase_run-1_bold.nii.gz
    name = re.sub(r"_+", "_", name).strip("_")         
    
    # MacOS versteckte Dateien entfernen
    if name.startswith("._"):
        name = name[2:] # MacOS versteckte Dateien entfernen
    return name

In [ ]:
# Migration für func eines Subjekts
def migrate_func_one_subject(old_sub="sub-00001", old_ses="ses-00001", dry_run=False):
    """Migriert die funktionellen Daten eines Subjekts von OLD_ROOT zu NEW_ROOT."""
    old_func = OLD_ROOT / old_sub / old_ses / "func"
    if not old_func.is_dir(): # Prüfen, ob der Pfad existiert
        raise FileNotFoundError(f"Alter func-Pfad nicht gefunden: {old_func}")
    
    new_sub = short_sub(old_sub) # sub-00001 -> 01
    new_ses = short_ses(old_ses) # ses-00001 -> ses-1
    new_func = NEW_ROOT / new_sub / new_ses / "func" # Neuer func-Pfad

    # func-Ordner anlegen
    os.makedirs(new_func, exist_ok=True)

    # Derivatives-Ordner anlegen
    os.makedirs(NEW_ROOT / "derivatives", exist_ok=True)

    # Alle Dateien im alten func-Ornder durchgehen
    for src in sorted(old_func.iterdir()):
        if not src.is_file(): # Nur Dateien (keine Ordner)
            continue
        #physio-Dateien, macOS-Datein, PAR-Dateien überspringen
        if "physio" in src.name or src.name.startswith("._") or src.name.lower().endswith(".par"):
            continue
        
        # Neuen Dateinamen erstellen
        dst_name = rename_func(src.name, new_sub, new_ses)
        dst = new_func / dst_name # Neuer Pfad

        #Kopieren
        if dry_run:
            print(f"[DRY] {src} -> {dst}")
        else:
            shutil.copy(src, dst) # Datei kopieren
            print(f"Kopiert: {src.name} -> {dst.relative_to(NEW_ROOT)}")

    print(f"Fertig. Zielordner: {new_func}")

In [ ]:
#Aufruf für sub-00001 ses-00001
migrate_func_one_subject(old_sub="sub-00001", old_ses="ses-00001", dry_run=False)

Kopiert: sub-00001_ses-00001_task-fmri_acq-10_rec-1_run-6_bold.json -> sub-01/ses-1/func/sub-01_ses-1_task-chase_run-6_bold.json
Kopiert: sub-00001_ses-00001_task-fmri_acq-10_rec-1_run-6_bold.nii.gz -> sub-01/ses-1/func/sub-01_ses-1_task-chase_run-6_bold.nii.gz
Kopiert: sub-00001_ses-00001_task-fmri_acq-3_rec-1_run-1_bold.json -> sub-01/ses-1/func/sub-01_ses-1_task-chase_run-1_bold.json
Kopiert: sub-00001_ses-00001_task-fmri_acq-3_rec-1_run-1_bold.nii.gz -> sub-01/ses-1/func/sub-01_ses-1_task-chase_run-1_bold.nii.gz
Kopiert: sub-00001_ses-00001_task-fmri_acq-4_rec-1_run-2_bold.json -> sub-01/ses-1/func/sub-01_ses-1_task-chase_run-2_bold.json
Kopiert: sub-00001_ses-00001_task-fmri_acq-4_rec-1_run-2_bold.nii.gz -> sub-01/ses-1/func/sub-01_ses-1_task-chase_run-2_bold.nii.gz
Kopiert: sub-00001_ses-00001_task-fmri_acq-6_rec-1_run-3_bold.json -> sub-01/ses-1/func/sub-01_ses-1_task-chase_run-3_bold.json
Kopiert: sub-00001_ses-00001_task-fmri_acq-6_rec-1_run-3_bold.nii.gz -> sub-01/ses-1/func/

In [ ]:
"""
# Im Zielordner nachträglich aufräumen bzw. umbenennen
Folder = Path("/mnt_01/ds-ASD/sub-01/ses-1/func")

for p in sorted(Folder.iterdir()):
    if not p.is_file():
        continue
    
    new = p.name

    #acq-* entfernen
    new = re.sub(r"_acq-[^_\.]+", "", new)
    
    #rec oder rec-* entfernen
    new = re.sub(r"_rec(?:-[^_\.]+)?", "", new)
    
    #run- wieder einsetzen, falls nach task- direkt eine Zahl kommt
    new = re.sub(r"(task-[^_]+)-(\d+)(?=_)", r"\1_run-\2", new)

    # Mehrfach-Underschores glätten
    new = re.sub(r"_+", "_", new).strip("_")

    if new != p.name:
        p.rename(p.with_name(new))
        print(f"{p.name} -> {new}")
        """

sub-01_ses-1_task-chase-1_bold.json -> sub-01_ses-1_task-chase_run-1_bold.json
sub-01_ses-1_task-chase-1_bold.nii.gz -> sub-01_ses-1_task-chase_run-1_bold.nii.gz
sub-01_ses-1_task-chase-1_bold.par -> sub-01_ses-1_task-chase_run-1_bold.par
sub-01_ses-1_task-chase-2_bold.json -> sub-01_ses-1_task-chase_run-2_bold.json
sub-01_ses-1_task-chase-2_bold.nii.gz -> sub-01_ses-1_task-chase_run-2_bold.nii.gz
sub-01_ses-1_task-chase-2_bold.par -> sub-01_ses-1_task-chase_run-2_bold.par
sub-01_ses-1_task-chase-3_bold.json -> sub-01_ses-1_task-chase_run-3_bold.json
sub-01_ses-1_task-chase-3_bold.nii.gz -> sub-01_ses-1_task-chase_run-3_bold.nii.gz
sub-01_ses-1_task-chase-3_bold.par -> sub-01_ses-1_task-chase_run-3_bold.par
sub-01_ses-1_task-chase-4_bold.json -> sub-01_ses-1_task-chase_run-4_bold.json
sub-01_ses-1_task-chase-4_bold.nii.gz -> sub-01_ses-1_task-chase_run-4_bold.nii.gz
sub-01_ses-1_task-chase-4_bold.par -> sub-01_ses-1_task-chase_run-4_bold.par
sub-01_ses-1_task-chase-5_bold.json -> sub-0

fmap

In [8]:
def rename_fmap(name: str, sub_new: str, ses_new: str) -> str:
    """Ersetzt die Subjekt- und Session-IDs im Dateinamen."""
    
    name = re. sub(r"sub-\d+", sub_new, name)
    name = re.sub(r"ses-\d+", ses_new, name)

    # _run-0 komplett entfernen
    name = re.sub(r"_run-0(?=[_.])", "", name)
                  
    #doppelte underscores glätten
    name = re.sub(r"_+", "_", name).strip("_")

    if name.startswith("._"):
        name = name[2:] # MacOS versteckte Dateien entfernen
    return name

In [9]:
# Migration für fmap eines Subjekts
def migrate_fmap_one_subject(old_sub="sub-00001", old_ses="ses-00001", dry_run=False):
    
    """Migriert die fmap-Daten eines Subjekts von OLD_ROOT zu NEW_ROOT."""
    old_fmap = OLD_ROOT / old_sub / old_ses / "fmap"
    if not old_fmap.is_dir(): # Prüfen, ob der Pfad existiert
        raise FileNotFoundError(f"Alter fmap-Pfad nicht gefunden: {old_fmap}")
    
    new_sub = short_sub(old_sub) # sub-00001 -> 01
    new_ses = short_ses(old_ses) # ses-00001 -> ses-1
    new_fmap = NEW_ROOT / new_sub / new_ses / "fmap" # Neuer fmap-Pfad

    # fmap-Ordner anlegen
    os.makedirs(new_fmap, exist_ok=True)
    os.makedirs(NEW_ROOT / "derivatives", exist_ok=True)

    # Alle Dateien im alten fmap-Ornder durchgehen
    for src in sorted(old_fmap.iterdir()):
        if not src.is_file(): # Nur Dateien (keine Ordner)
            continue
        #physio-Dateien bhealten, nur macOS-Noise überspringen
        if src.name.startswith("._"):
            continue
        
        # Neuen Dateinamen erstellen
        dst_name = rename_fmap(src.name, new_sub, new_ses)
        dst = new_fmap / dst_name # Neuer Pfad

        #Kopieren
        if dry_run:
            print(f"[DRY] {src} -> {dst}")
        else:
            shutil.copy(src, dst) # Datei kopieren
            print(f"Kopiert: {src.name} -> {dst.relative_to(NEW_ROOT)}")

    print(f"Fertig (fmap). Zielordner: {new_fmap}")

In [ ]:
migrate_fmap_one_subject(old_sub="sub-00001", old_ses="ses-00001", dry_run=False)
    # (und falls noch nicht gelaufen:)
    # migrate_func_one_subject(old_sub="sub-00001", old_ses="ses-00001", dry_run=False)


Kopiert: sub-00001_ses-00001_acq-2_run-0_magnitude1.nii.gz -> sub-01/ses-1/fmap/sub-01_ses-1_acq-2_magnitude1.nii.gz
Kopiert: sub-00001_ses-00001_acq-2_run-0_magnitude2.nii.gz -> sub-01/ses-1/fmap/sub-01_ses-1_acq-2_magnitude2.nii.gz
Kopiert: sub-00001_ses-00001_acq-2_run-0_phase1.json -> sub-01/ses-1/fmap/sub-01_ses-1_acq-2_phase1.json
Kopiert: sub-00001_ses-00001_acq-2_run-0_phase1.nii.gz -> sub-01/ses-1/fmap/sub-01_ses-1_acq-2_phase1.nii.gz
Kopiert: sub-00001_ses-00001_acq-2_run-0_phase1.par -> sub-01/ses-1/fmap/sub-01_ses-1_acq-2_phase1.par
Kopiert: sub-00001_ses-00001_acq-2_run-0_phase2.json -> sub-01/ses-1/fmap/sub-01_ses-1_acq-2_phase2.json
Kopiert: sub-00001_ses-00001_acq-2_run-0_phase2.nii.gz -> sub-01/ses-1/fmap/sub-01_ses-1_acq-2_phase2.nii.gz
Kopiert: sub-00001_ses-00001_acq-2_run-0_recording-scanphys_physio.json -> sub-01/ses-1/fmap/sub-01_ses-1_acq-2_recording-scanphys_physio.json
Kopiert: sub-00001_ses-00001_acq-2_run-0_recording-scanphys_physio.log -> sub-01/ses-1/fmap/

anat

In [14]:
def rename_anat(name: str, sub_new: str, ses_new: str) -> str:
    """Ersetzt die Subjekt- und Session-IDs im Dateinamen."""
    
    # sub-00001 -> sub-01
    name = re. sub(r"sub-\d+", sub_new, name)

    # ses-00001 -> ses-1
    name = re.sub(r"ses-\d+", ses_new, name)

    # acq-* entfernen
    name = re.sub(r"_acq-[^_\.]+", "", name)

    # rec/rec-* entfernen
    name = re.sub(r"_rec(?:-[^_\.]+)?", "", name)

    # Underscores glätten
    name = re.sub(r"_+", "_", name).strip("_")   

    # macOS versteckte Dateien entfernen
    if name.startswith("._"):
        name = name[2:] # MacOS versteckte Dateien entfernen
    return name

In [15]:
# Migration für anat eines Subjekts
def migrate_anat_one_subject(old_sub="sub-00001", old_ses="ses-00001", dry_run=False):
    """Migriert die anatomischen Daten (anat) eines Subjekts.
    """
    old_anat = OLD_ROOT / old_sub / old_ses / "anat"
    if not old_anat.is_dir():
        raise FileNotFoundError(f"Alter anat-Pfad nicht gefunden: {old_anat}")

    new_sub = short_sub(old_sub)     # z.B. sub-01
    new_ses = short_ses(old_ses)     # z.B. ses-1
    new_anat = NEW_ROOT / new_sub / new_ses / "anat"
    os.makedirs(new_anat, exist_ok=True)
    os.makedirs(NEW_ROOT / "derivatives", exist_ok=True)

    # Nur bestimmte Dateitypen behalten
    pat_json = re.compile(r"_T1w\.json$", re.IGNORECASE) # JSON-Dateien für T1w behalten
    pat_nii = re.compile(r"_T1w\.nii(?:\.gz)?$", re.IGNORECASE) # NIfTI-Dateien für T1w behalten

    for src in sorted(old_anat.iterdir()):
        if not src.is_file():
            continue
        if src.name.startswith("._"):   # macOS Noise
            continue
        if not (pat_json.search(src.name) or pat_nii.search(src.name)):
            continue  # alles Nicht-T1w automatisch raus

        dst_name = rename_anat(src.name, new_sub, new_ses)
        dst = new_anat / dst_name

        if dry_run:
            print(f"[DRY] {src} -> {dst}")
        else:
            shutil.copy(src, dst)
            print(f"Kopiert: {src.relative_to(OLD_ROOT)} -> {dst.relative_to(NEW_ROOT)}")

    print(f"Fertig (anat). Zielordner: {new_anat}")

In [ ]:
migrate_anat_one_subject(old_sub="sub-00001", old_ses="ses-00001", dry_run=False)


Kopiert: sub-00001/ses-00001/anat/sub-00001_ses-00001_acq-11_rec-1_T1w.json -> sub-01/ses-1/anat/sub-01_ses-1_T1w.json
Kopiert: sub-00001/ses-00001/anat/sub-00001_ses-00001_acq-11_rec-1_T1w.nii.gz -> sub-01/ses-1/anat/sub-01_ses-1_T1w.nii.gz
Fertig (anat). Zielordner: /mnt_01/ds-ASD/sub-01/ses-1/anat


Für alle Subjekte

In [21]:
# Gesamt-Loop über alle Subjekte und Sessions
def migrate_all_subjects(dry_run=False):
    """Alle Subjekte und Sessions unter OLD_ROOT durchsuchen und migrieren inkl. anat, fmap, func.
    """

    subs = sorted(p for p in OLD_ROOT.iterdir() if p.is_dir() if p. is_dir() and p.name.startswith("sub-"))
    if not subs:
        print(f"Keine Subjekte im alten BIDS-Pfad gefunden unter: {OLD_ROOT}")
        return
    
    n_sub = n_ses = moved_anat = moved_fmap = moved_func = 0

    for sub_path in subs:
        old_sub = sub_path.name
        n_sub += 1

        ses_paths = sorted(p for p in sub_path.iterdir() if p.is_dir() and p.name.startswith("ses-"))
        if not ses_paths:
            print(f"Keine Sessions für {old_sub} gefunden, übersprungen...")
            continue

        for ses_path in ses_paths:    
            old_ses = ses_path.name
            n_ses += 1
            print(f"\n=== {old_sub}/{old_ses} ===")

        # ANAT
            try:
                migrate_anat_one_subject(old_sub=old_sub, old_ses=old_ses, dry_run=dry_run)
                moved_anat += 1
            except FileNotFoundError:
                print(f"… anat: kein Ordner, übersprungen.")

            # FMAP
            try:
                migrate_fmap_one_subject(old_sub=old_sub, old_ses=old_ses, dry_run=dry_run)
                moved_fmap += 1
            except FileNotFoundError:
                print(f"… fmap: kein Ordner, übersprungen.")

            # FUNC
            try:
                migrate_func_one_subject(old_sub=old_sub, old_ses=old_ses, dry_run=dry_run)
                moved_func += 1
            except FileNotFoundError:
                print(f"… func: kein Ordner, übersprungen.")

    print("\n---------------- SUMMARY ----------------")
    print(f"Subjekte gesamt:     {n_sub}")
    print(f"Sessionen gesamt:    {n_ses}")
    print(f"Anat migriert:       {moved_anat}")
    print(f"Fmap migriert:       {moved_fmap}")
    print(f"Func migriert:       {moved_func}")
    print("-----------------------------------------")


In [ ]:
migrate_all_subjects(dry_run=False)


=== sub-00001/ses-00001 ===
Kopiert: sub-00001/ses-00001/anat/sub-00001_ses-00001_acq-11_rec-1_T1w.json -> sub-01/ses-1/anat/sub-01_ses-1_T1w.json
Kopiert: sub-00001/ses-00001/anat/sub-00001_ses-00001_acq-11_rec-1_T1w.nii.gz -> sub-01/ses-1/anat/sub-01_ses-1_T1w.nii.gz
Fertig (anat). Zielordner: /mnt_01/ds-ASD/sub-01/ses-1/anat
Kopiert: sub-00001_ses-00001_acq-2_run-0_magnitude1.nii.gz -> sub-01/ses-1/fmap/sub-01_ses-1_acq-2_magnitude1.nii.gz
Kopiert: sub-00001_ses-00001_acq-2_run-0_magnitude2.nii.gz -> sub-01/ses-1/fmap/sub-01_ses-1_acq-2_magnitude2.nii.gz
Kopiert: sub-00001_ses-00001_acq-2_run-0_phase1.json -> sub-01/ses-1/fmap/sub-01_ses-1_acq-2_phase1.json
Kopiert: sub-00001_ses-00001_acq-2_run-0_phase1.nii.gz -> sub-01/ses-1/fmap/sub-01_ses-1_acq-2_phase1.nii.gz
Kopiert: sub-00001_ses-00001_acq-2_run-0_phase1.par -> sub-01/ses-1/fmap/sub-01_ses-1_acq-2_phase1.par
Kopiert: sub-00001_ses-00001_acq-2_run-0_phase2.json -> sub-01/ses-1/fmap/sub-01_ses-1_acq-2_phase2.json
Kopiert: sub-